In [126]:
import pandas as pd
from datetime import datetime
from datetime import date

### Consolidación de información por nivel de sector
Debemos consolidar las hojas(bases de datos) correspondientes a ROTURAS y FUGAS en una sola tabla que facilite la construcción de indicadores operativos.

In [ ]:
#Coombinar dos hojas de una misma fuente

df_roturas = pd.read_excel("../database_clean/02_ROTURAS_RED_PRIMARIA_SECUNDARIA_SAN_MARTIN_CLEAN.xlsx", sheet_name="ROTURAS_RED_PRIMARIA")

df_fugas = pd.read_excel("../database_clean/02_ROTURAS_RED_PRIMARIA_SECUNDARIA_SAN_MARTIN_CLEAN.xlsx", sheet_name="FUGAS_RED_SECUNDARIA")

consolidado = pd.concat([df_roturas, df_fugas])

In [ ]:
# Combinar hoja de redes de conexión a nuestra base de datos consolidada


In [118]:


#transformamos campo fecha a pd datetime
consolidado["fecha_inicio"] = pd.to_datetime(consolidado['fecha_inicio']).dt.date
consolidado["fecha_fin"] = pd.to_datetime(consolidado['fecha_fin']).dt.date

consolidado.sample(5)

,CONEXION,SECTOR,CDESTIPSER,MES,AÑO,CATEGORIA,fecha_inicio,hora_inicio,fecha_fin,hora_fin,latitud,longitud
10427,1061370,S58,AG: Conexiones Domiciliarias ...,10,2025,Fuga,2025-10-06,1960-01-01 17:30:00,2025-10-07,1960-01-01 09:11:25.280,-6.481452,-76.382828
7894,1197252,S63,AG: Conexiones Domiciliarias ...,5,2025,Fuga,2025-05-25,1960-01-01 02:10:00,2025-05-26,1960-01-01 21:47:56.296,-6.475916,-76.369514
4026,2381124,S42,AG: Conexiones Domiciliarias ...,4,2025,Fuga,2025-04-27,1960-01-01 01:59:00,2025-04-28,1960-01-01 14:43:28.580,-6.495295,-76.387625
513,1307580,S26,AG: Conexiones Domiciliarias ...,5,2025,Rotura Tuberia Matriz,2025-05-28,1960-01-01 14:18:00,2025-05-29,1960-01-01 16:30:57.480,-6.527593,-76.384684
426,1335205,S64,AG: Conexiones Domiciliarias ...,9,2025,Fuga,2025-09-25,1960-01-01 03:06:00,2025-09-27,1960-01-01 01:12:34.533,-6.485759,-76.374467


In [125]:

#Se crea un dataframe basado en los sectores únicos obtenidos
df_reto2 = pd.DataFrame({"SECTOR": consolidado["SECTOR"].unique()})

#obtenemos el total de roturas para cada sector
df_reto2["TOTAL ROTURAS"] = df_reto2["SECTOR"].apply(lambda x: consolidado[(consolidado["SECTOR"] == x) & (consolidado["CATEGORIA"] == "Rotura Tuberia Matriz")].shape[0])
#obtenemos el total de fugas para cada sector
df_reto2["TOTAL FUGAS"] = df_reto2["SECTOR"].apply(lambda x: consolidado[(consolidado["SECTOR"] == x) & (consolidado["CATEGORIA"] == "Fuga")].shape[0])

#Obtenemos el tiempo promedio tomado en la reparación de fugas por cada sector
"""
Consideramos que existen 4 columnas de interés:
fecha_inicio - hora_inicio
fecha_fin - hora_fin

La idea es restar las columnas respectivas para los registros asociados a cada sector y luego promediarlo
"""

def tiempo_reparacion_fugas(sector):
    #filtramos el dataframe para obtener solo las filas asociadas al sector y a la categoría de fuga
    df_fugas_sector = consolidado[(consolidado["SECTOR"] == sector) & (consolidado["CATEGORIA"] == "Fuga")]
    
    #Tiempo de reparación para cada registro
    df_fugas_sector["tiempo_reparacion"] = (pd.to_datetime(df_fugas_sector["fecha_fin"]) - pd.to_datetime(df_fugas_sector["fecha_inicio"])).dt.total_seconds() / 3600
    
    #Calculamos lamedia por sector
    tiempo_promedio = df_fugas_sector["tiempo_reparacion"].mean()
    
    return tiempo_promedio

df_reto2["TIEMPO PROMEDIO REPARACION FUGAS (HORAS)"] = df_reto2["SECTOR"].apply(tiempo_reparacion_fugas)

def tiempo_reparacion(sector):
    df_fugas_sector = consolidado[(consolidado["SECTOR"] == sector)]
    
    df_fugas_sector["tiempo_reparacion"] = (pd.to_datetime(df_fugas_sector["fecha_fin"]) - pd.to_datetime(df_fugas_sector["fecha_inicio"])).dt.total_seconds() / 3600
    
    tiempo_promedio = df_fugas_sector["tiempo_reparacion"].mean()
    
    return tiempo_promedio


df_reto2["TIEMPO PROMEDIO REPARACION DE SECTOR (HORAS)"] = df_reto2["SECTOR"].apply(tiempo_reparacion)
df_reto2

,SECTOR,TOTAL ROTURAS,TOTAL FUGAS,TIEMPO PROMEDIO REPARACION FUGAS (HORAS),TIEMPO PROMEDIO REPARACION DE SECTOR (HORAS)
0,S74,21,214,26.915888,29.310638
1,S72,12,188,25.787234,27.600000
2,S64,8,164,26.487805,27.488372
3,S76,17,309,25.320388,26.944785
4,S49,6,129,27.534884,28.444444
...,...,...,...,...,...
77,S21,0,76,27.789474,27.789474
78,S9,0,30,26.400000,26.400000
79,S1,0,23,29.217391,29.217391
80,S5,0,3,32.000000,32.000000
